**Table of contents**<a id='toc0_'></a>    
- 1. [Setup](#toc1_)    
  - 1.1. [Install Dependencies](#toc1_1_)    
  - 1.2. [Import Libraries](#toc1_2_)    
  - 1.3. [Check Runtime / Device](#toc1_3_)    
- 2. [Load Dataset](#toc2_)    
  - 2.1. [Find Project Root](#toc2_1_)    
  - 2.2. [Read Validation Data](#toc2_2_)    
- 3. [Configuration](#toc3_)    
- 4. [Load Qwen Model](#toc4_)    
- 5. [Prompting & Inference](#toc5_)    
  - 5.1. [Build Prompt](#toc5_1_)    
  - 5.2. [Generate Summary](#toc5_2_)    
  - 5.3. [Test One Sample](#toc5_3_)    
- 6. [Evaluate Qwen](#toc6_)    
  - 6.1. [Generate Predictions](#toc6_1_)    
  - 6.2. [Compute ROUGE](#toc6_2_)    
  - 6.3. [Optional: Compute BERTScore](#toc6_3_)    
- 7. [Save Outputs](#toc7_)  

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[Setup](#toc0_)


## 1.1. <a id='toc1_1_'></a>[Install Dependencies](#toc0_)

In [ ]:
# Run this cell once if your environment does not have the required packages.
# In VS Code, make sure the selected kernel is the project's virtual environment.

import sys

!{sys.executable} -m pip install -q -U     pandas pyarrow tqdm     transformers accelerate sentencepiece     evaluate rouge-score bert-score

## 1.2. <a id='toc1_2_'></a>[Import Libraries](#toc0_)

In [ ]:
import os
import gc
import time
from pathlib import Path
from dataclasses import dataclass

import pandas as pd
from tqdm.auto import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import evaluate

## 1.3. <a id='toc1_3_'></a>[Check Runtime / Device](#toc0_)

In [ ]:
print("Python executable:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
else:
    print("No CUDA GPU detected. Qwen inference on CPU will be very slow.")

# 2. <a id='toc2_'></a>[Load Dataset](#toc0_)

## 2.1. <a id='toc2_1_'></a>[Find Project Root](#toc0_)

In [ ]:
PROJECT_ROOT = Path.cwd()

# If this notebook is opened from the notebooks/ folder, move one level up.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)

## 2.2. <a id='toc2_2_'></a>[Read Validation Data](#toc0_)

In [5]:
DATA_PATH = PROJECT_ROOT / "data" / "valid_data.parquet"

# Optional fallback: allow testing with a small sample file if full validation data is not available.
SAMPLE_DATA_PATH = PROJECT_ROOT / "data" / "sample_valid.parquet"

if DATA_PATH.exists():
    data_path = DATA_PATH
elif SAMPLE_DATA_PATH.exists():
    data_path = SAMPLE_DATA_PATH
else:
    raise FileNotFoundError(
        "Cannot find validation data. Please place valid_data.parquet in data/ "
        "or provide data/sample_valid.parquet for quick testing."
    )

print("Using data file:", data_path)

df = pd.read_parquet(data_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Using data file: f:\NLP\NLP-Abstractive-Summary\data\valid_data.parquet
Shape: (1349, 2)
Columns: ['article', 'summary']


,article,summary
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ..."
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...


# 3. <a id='toc3_'></a>[Configuration](#toc0_)


In [6]:
@dataclass
class EvalConfig:
    # Use 0.5B by default for local VS Code testing.
    # Change to "Qwen/Qwen2.5-1.5B-Instruct" if you have enough GPU memory.
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"

    # Dataset columns
    text_col: str = "article"
    summary_col: str = "summary"

    # Inference settings
    max_input_chars: int = 3000
    max_token_length: int = 2048
    max_new_tokens: int = 128

    # Evaluation settings. Keep small for local testing.
    max_samples: int = 10

    # Deterministic decoding for reproducible evaluation.
    do_sample: bool = False
    num_beams: int = 1
    repetition_penalty: float = 1.1


config = EvalConfig()
print(config)

EvalConfig(model_name='Qwen/Qwen2.5-0.5B-Instruct', text_col='article', summary_col='summary', max_input_chars=3000, max_token_length=2048, max_new_tokens=128, max_samples=10, do_sample=False, num_beams=1, repetition_penalty=1.1)


In [7]:
# Keep only required columns and remove missing rows.
# This prevents NameError problems from undefined TEXT_COL/SUMMARY_COL variables.

TEXT_COL = config.text_col
SUMMARY_COL = config.summary_col

required_cols = [TEXT_COL, SUMMARY_COL]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing columns in dataset: {missing_cols}. Current columns: {df.columns.tolist()}")

df = df[required_cols].dropna().reset_index(drop=True)

print("Cleaned shape:", df.shape)
df.head()

Cleaned shape: (1349, 2)


,article,summary
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ..."
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...


# 4. <a id='toc4_'></a>[Load Qwen Model Locally](#toc0_)

In [ ]:
MODEL_NAME = config.model_name

print("Loading tokenizer:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# Use float16 on CUDA to reduce VRAM usage.
# Use float32 on CPU because float16 CPU inference may be unstable/slow.
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("Loading model:", MODEL_NAME)
print("Torch dtype:", torch_dtype)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

# If running on CPU, explicitly move model to CPU.
if not torch.cuda.is_available():
    model = model.to("cpu")

model.eval()
print("Model loaded successfully.")

# 5. <a id='toc5_'></a>[Prompting & Inference](#toc0_)

## 5.1. <a id='toc5_1_'></a>[Build Prompt](#toc0_)

In [9]:
def build_prompt(article: str) -> str:
    # Build instruction prompt for Vietnamese abstractive summarization.
    return f"""
You are a Vietnamese abstractive summarization system.

Summarize the following Vietnamese article in Vietnamese.
Do not copy long sentences directly.
Keep the main ideas, important facts, names, places, numbers, and conclusions.
Write a concise and natural summary.

Article:
{article}

Summary:
""".strip()

## 5.2. <a id='toc5_2_'></a>[Generate Summary](#toc0_)

In [10]:
def generate_qwen_summary(article: str) -> str:
    # Convert to string and truncate very long article for local testing.
    article = str(article)[:config.max_input_chars]

    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant specialized in Vietnamese abstractive summarization."
        },
        {
            "role": "user",
            "content": build_prompt(article)
        }
    ]

    # Qwen Instruct models expect chat template formatting.
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=config.max_token_length
    )

    # Move input tensors to the same device as the model.
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=config.max_new_tokens,
            do_sample=config.do_sample,
            num_beams=config.num_beams,
            repetition_penalty=config.repetition_penalty,
            pad_token_id=tokenizer.eos_token_id
        )

    # Only decode newly generated tokens, not the original prompt.
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    summary = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return summary.strip()

## 5.3. <a id='toc5_3_'></a>[Test One Sample](#toc0_)


In [12]:
sample = df.iloc[0]

article = sample[TEXT_COL]
reference = sample[SUMMARY_COL]

prediction = generate_qwen_summary(article)

print("ARTICLE:")
print(article[:1000])

print("REFERENCE SUMMARY:")
print(reference)

print("QWEN SUMMARY:")
print(prediction)

ARTICLE:
Giải thưởng công bố gần đây bởi World Travel Awards. Đây là năm thứ hai liên tiếp InterContinental Phu Quoc Long Beach Resort được vinh danh ở hạng mục gia đình trên toàn châu Á. Khu nghỉ dưỡng tọa lạc bên biển Phú Quốc, nổi bật với thiết kế lấy cảm hứng từ đại dương. Khuôn viên rộng rãi với nhiều mảng xanh thiên nhiên đậm chất nhiệt đới. Không gian sảnh lễ tân, phòng nghỉ, villa, nhà hàng... đều được chú trọng để tạo sự hài hòa với biển và cây cối. Du khách có thể chọn nghỉ ngơi tại khu phòng khách sạn rộng rãi, tiện nghi, hoặc những căn hộ, phòng suite, biệt thự cao cấp hướng biển. Mỗi không gian được thiết kế dựa trên tinh thần gắn kết các thành viên trong gia đình. Bên cạnh tiện nghi sang trọng, InterContinental Phu Quoc còn hút khách gia đình nhờ loạt trải nghiệm giải trí, thư giãn đa dạng, phù hợp với mọi độ tuổi. Với trẻ nhỏ, khu Planet Trekker là nơi các bé có thể thoải mái vui chơi, học hỏi từ các đầu sách thiếu nhi, những buổi workshop thủ công... Phụ huynh có thể yê

# 6. <a id='toc6_'></a>[Evaluate Qwen](#toc0_)


## 6.1. <a id='toc6_1_'></a>[Generate Predictions](#toc0_)

In [13]:
test_df = df.head(config.max_samples).copy()

predictions = []
references = []
articles = []

start_time = time.time()

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating summaries"):
    article = row[TEXT_COL]
    reference = str(row[SUMMARY_COL])

    try:
        pred = generate_qwen_summary(article)
    except RuntimeError as e:
        # Common case: CUDA out of memory.
        print("RuntimeError:", e)
        pred = ""

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
    except Exception as e:
        # Keep evaluation running even if one sample fails.
        print("Error:", e)
        pred = ""

    articles.append(article)
    references.append(reference)
    predictions.append(pred)

elapsed = time.time() - start_time
print(f"Generated {len(predictions)} summaries in {elapsed:.2f} seconds")

Generating summaries: 100%|██████████| 10/10 [02:29<00:00, 14.99s/it]

Generated 10 summaries in 149.89 seconds


In [14]:
result_df = pd.DataFrame({
    "article": articles,
    "reference_summary": references,
    "qwen_summary": predictions
})

result_df.head()

,article,reference_summary,qwen_summary
0,Giải thưởng công bố gần đây bởi World Travel A...,InterContinental Phu Quoc Long Beach Resort đã...,"InterContinental Phu Quoc Long Beach Resort, m..."
1,Theo bảng xếp hạng 20 quốc gia tốt nhất thế gi...,Việt Nam đã xếp hạng 15 trên bảng xếp hạng 20 ...,Để xếp hạng 20 quốc gia tốt nhất thế giới năm ...
2,"Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ...","Ngày hội Văn hóa, Thể thao và Du lịch các dân ..."
3,Giải thưởng do Tạp chí du lịch Condé Nast Trav...,"Phú Quốc, đảo ngọc của Việt Nam, đã được vinh ...",Đại diện cho tạp chí du lịch Condé Nast Travel...
4,KKday Vietnam vừa công bố hợp tác chiến lược S...,KKday Vietnam vừa công bố hợp tác chiến lược v...,Đây là một sự hợp tác chiến lược giữa KKday và...


## 6.2. <a id='toc6_2_'></a>[Compute ROUGE](#toc0_)

In [15]:
# Remove empty predictions before computing metrics.
valid_pairs = [
    (pred, ref)
    for pred, ref in zip(predictions, references)
    if isinstance(pred, str) and pred.strip()
]

if not valid_pairs:
    raise ValueError("No valid predictions found. Please check model inference errors above.")

valid_predictions, valid_references = zip(*valid_pairs)

rouge = evaluate.load("rouge")

rouge_scores = rouge.compute(
    predictions=list(valid_predictions),
    references=list(valid_references),
    use_stemmer=False
)

rouge_scores

{'rouge1': np.float64(0.6920784261023987),
 'rouge2': np.float64(0.4272152299667006),
 'rougeL': np.float64(0.4276240312864963),
 'rougeLsum': np.float64(0.43015304922412095)}

## 6.3. <a id='toc6_3_'></a>[Optional: Compute BERTScore](#toc0_)

BERTScore đánh giá mức độ tương đồng ngữ nghĩa tốt hơn ROUGE, đặc biệt khi summary không dùng đúng từ như reference.

In [ ]:
# Optional metric. Run this cell if you want semantic similarity score.
# For Vietnamese, lang="vi" is usually acceptable. If it fails, try model_type="xlm-roberta-large".

bertscore = evaluate.load("bertscore")

bert_scores = bertscore.compute(
    predictions=list(valid_predictions),
    references=list(valid_references),
    lang="vi"
)

bert_precision = sum(bert_scores["precision"]) / len(bert_scores["precision"])
bert_recall = sum(bert_scores["recall"]) / len(bert_scores["recall"])
bert_f1 = sum(bert_scores["f1"]) / len(bert_scores["f1"])

print("BERTScore Precision:", bert_precision)
print("BERTScore Recall:", bert_recall)
print("BERTScore F1:", bert_f1)

# 7. <a id='toc7_'></a>[Save Outputs](#toc0_)

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

output_file = OUTPUT_DIR / f"qwen_predictions_{config.max_samples}_samples.csv"

result_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("Saved predictions to:", output_file)

In [ ]:
# Save metrics as a small CSV file for report writing.
metrics = {
    "model_name": config.model_name,
    "num_samples": len(valid_predictions),
    "rouge1": rouge_scores.get("rouge1"),
    "rouge2": rouge_scores.get("rouge2"),
    "rougeL": rouge_scores.get("rougeL"),
    "rougeLsum": rouge_scores.get("rougeLsum"),
}

# Add BERTScore if the optional cell was executed.
if "bert_f1" in globals():
    metrics.update({
        "bertscore_precision": bert_precision,
        "bertscore_recall": bert_recall,
        "bertscore_f1": bert_f1,
    })

metrics_df = pd.DataFrame([metrics])
metrics_file = OUTPUT_DIR / f"qwen_metrics_{config.max_samples}_samples.csv"

metrics_df.to_csv(metrics_file, index=False, encoding="utf-8-sig")

print("Saved metrics to:", metrics_file)
metrics_df